# Background-relative beta-effect analysis

This notebook tests whether EAC anticyclonic eddies (AEs) have a northward/equatorward residual drift and cyclonic eddies (CEs) a southward/poleward residual drift after subtracting the strong southwestward background flow. It uses the cache produced by `build_background_cache.py`.

The primary sample has planetary PV-gradient magnitude greater than topographic PV-gradient magnitude. The topographic term uses $(w+f)\nabla h/h^2$, so the dominance mask is calculated separately for AEs and CEs.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / 'background_flow_tools.py').exists():
    HERE = Path('MRes/seacofs_eddy_tilt_analysis/beta_effect_background_flow').resolve()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent))

import seacofs_tilt_tools as tilt
from background_flow_tools import BackgroundConfig, load_background_cache
from analysis_tools import (METHODS, add_track_velocity, add_residual_velocities,
                            propagation_summary, eddy_bootstrap_ci,
                            vorticity_budget_summary)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load eddies and polarity-specific PV-gradient terms

`topo_plan_ratio = log(|topographic PV gradient| / |planetary PV gradient|)`. Therefore values below zero identify planetary-dominant observations. Because the topographic numerator contains `w + f`, AE and CE masks need not select the same locations.

In [2]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = tilt.add_pv_gradient_terms(df, grid)
primary = df.loc[df['topo_plan_ratio'] < 0].copy()
pd.crosstab(primary['Cyc'], columns='observations').join(
    primary.groupby('Cyc')['Eddy'].nunique().rename('eddies'))

,observations,eddies
Cyc,,
AE,13544,903
CE,3239,706


## 2. Load the one-pass background cache

Run `01_build_background_cache.ipynb` first. Six estimates are retained: instantaneous annulus surface flow (primary), instantaneous thickness-weighted 0–500 m annulus flow, monthly surface and 0–500 m climatologies, and exact count-weighted full-archive surface and 0–500 m means. The full-archive fields are additional sensitivity tests, not replacements for the seasonally resolved estimates.

In [3]:
config = BackgroundConfig(
    cache_root=Path(
        "/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/"
        "background_flow_cache_all_eddies_v3"
    )
)
background = load_background_cache(config)
required_full = {'full_surface_east_ms', 'full_surface_north_ms',
                 'full_500_east_ms', 'full_500_north_ms'}
missing_full = required_full - set(background.columns)
if missing_full:
    raise RuntimeError('Rerun 01_build_background_cache.ipynb once to add the full-archive means.')
primary = primary.merge(background, on=['Eddy', 'Day'], how='inner', validate='one_to_one')
primary = add_track_velocity(primary, grid.angle, window=5)
primary = add_residual_velocities(primary)
print(f"Matched {len(primary):,} observations from {primary['Eddy'].nunique():,} eddies")

RuntimeError: Rerun 01_build_background_cache.ipynb once to add the full-archive means.

## 3. Residual meridional propagation

Track velocities are obtained with a time-aware position derivative and then smoothed with a centred five-observation rolling median. The rotated-grid `xc`, `yc` velocities are converted to true east/north using the ROMS grid angle. Confidence intervals resample whole eddies, not individual days. Evidence for beta drift is AE residual north velocity above zero and CE residual north velocity below zero, robust across background definitions.

In [ ]:
summary = propagation_summary(primary, n_boot=5000)
display(summary)

plot = summary[summary['column'].str.contains('residual')].copy()
plot['method'] = plot['column'].str.replace('_residual_north_ms', '', regex=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (cyc, part) in zip(axes, plot.groupby('Cyc')):
    y = np.arange(len(part))
    ax.errorbar(part['median'], y, xerr=[part['median']-part['ci_low'], part['ci_high']-part['median']], fmt='o')
    ax.axvline(0, color='k', lw=1)
    ax.set(yticks=y, yticklabels=part['method'], title=cyc, xlabel='Residual north velocity (m s$^{-1}$)')
fig.suptitle('Planetary-dominant eddies: whole-eddy bootstrap estimates')
plt.tight_layout()

## 4. Vorticity-budget consistency check

For meridional residual velocity $v_r$, planetary-vorticity advection predicts $d w/dt \approx -\beta v_r$. This is a stronger mechanistic diagnostic than a simple association between vorticity and tilt. A budget residual centred near zero supports consistency, but does not alone establish causality.

In [ ]:
budget, budget_summary = vorticity_budget_summary(primary, method='ann_surface')
display(budget_summary)
for cyc, part in budget.groupby('Cyc'):
    x = part['beta_advection_s2']
    y = part['dw_dt_s2']
    use = np.isfinite(x) & np.isfinite(y)
    plt.hexbin(x[use], y[use], gridsize=35, mincnt=1, bins='log', label=cyc)
    plt.xlabel(r'Predicted $-\beta v_r$ (s$^{-2}$)')
    plt.ylabel(r'Observed $dw/dt$ (s$^{-2}$)')
    plt.title(cyc)
    plt.show()

## 5. Link residual propagation to meridional tilt

Positive aligned values mean equatorward for AEs and poleward for CEs. This section tests whether the expected residual drift and the observed tilt point in the same meridional sense. Interpret day-level correlations cautiously because repeated observations from one eddy are not independent.

In [ ]:
theta = np.deg2rad(primary['TiltDir'])
primary['tilt_north_km'] = primary['TiltDis'] * np.cos(theta)
primary['aligned_tilt_km'] = primary['expected_sign'] * primary['tilt_north_km']
rows = []
for method in METHODS:
    x = primary[f'{method}_aligned_residual_north_ms']
    rows.append({'method': method, 'spearman': x.corr(primary['aligned_tilt_km'], method='spearman'),
                 'n': int((x.notna() & primary['aligned_tilt_km'].notna()).sum())})
display(pd.DataFrame(rows))

## 6. PV-dominance sensitivity

Require progressively stronger planetary dominance. Since the cache was built for `topo_plan_ratio < 0`, thresholds here are planetary/topographic ratios of 1, 2, and 4. The polarity-specific `(w + f)` term remains in every mask.

In [ ]:
sensitivity = []
for ratio in [1, 2, 4]:
    part = primary[primary['topo_plan_ratio'] < -np.log(ratio)]
    result = eddy_bootstrap_ci(part, 'ann_surface_residual_north_ms', n_boot=5000)
    result['minimum_planetary_topographic_ratio'] = ratio
    sensitivity.append(result)
display(pd.concat(sensitivity, ignore_index=True))

## Interpretation checklist

The beta-effect interpretation becomes credible if: (1) AE and CE residual meridional velocities have the predicted opposite signs; (2) signs persist for instantaneous and climatological, surface and 0–500 m backgrounds; (3) the observed vorticity tendency is consistent with `-beta * residual north velocity`; (4) the signal strengthens under stricter planetary dominance; and (5) residual propagation aligns with the polarity-specific meridional tilt. Failure of one diagnostic is informative and should not be hidden by relying only on the vorticity-versus-tilt correlation.